# Superstore Dataset — Data Exploration & Cleaning (Pandas)

**Objective:** Learn Python basics and use Pandas to load, explore, and clean a CSV dataset.

> **Note:** This sandbox environment has no internet access, so the dataset could not be downloaded directly from Kaggle. Instead, a **synthetic dataset with the same structure as the Superstore dataset** was generated (same columns, missing values, and duplicate rows included on purpose so the cleaning steps have something real to work on). You can run this exact same notebook on your original Kaggle CSV — you'll just need to change the file name/path.

Below are all 7 steps, one at a time, with code and real output.

## Step 0: Import libraries

Pandas is the most popular Python library for working with tabular data (like Excel, but with code). Numpy helps with numeric operations.

In [1]:
import pandas as pd
import numpy as np

# Display settings so columns show up nicely
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("Pandas version:", pd.__version__)

Pandas version: 3.0.2

## Step 1: Load the CSV file into a Pandas DataFrame

`pd.read_csv()` reads a CSV file and converts it into a DataFrame (a table, similar to Excel).

In [1]:
# Put your CSV file path here
df = pd.read_csv("superstore_raw.csv")

# Look at the first 5 rows
df.head()

   Order_ID  Order_Date Customer_Name      Segment      City State         Category Sub_Category     Product_Name       Price  Quantity  Discount
0  ORD-1000  2023-08-17   Customer_18  Home Office   Houston    TX       Technology       Laptop   Laptop Model-4       17.38       5.0       0.3
1  ORD-1001  2023-04-06   Customer_28     Consumer  New York    NY       Technology       Laptop  Laptop Model-17      297.29       1.0       0.2
2  ORD-1002  2023-08-14   Customer_58  Home Office     Miami    FL        Furniture        Table    Table Model-9      359.43       9.0       0.0
3  ORD-1003  2023-12-11   Customer_14     Consumer   Houston    TX        Furniture         Sofa    Sofa Model-13      173.42       3.0       0.0
4  ORD-1004  2024-07-03   Customer_16    Corporate    Denver    CO  Office Supplies      Pen Set  Pen Set Model-3      303.84       1.0       0.2

## Step 2: Explore the data

- `.head()` / `.tail()` → first/last rows
- `.shape` → (rows, columns) count
- `.columns` → column names
- `.dtypes` → data type of each column
- `.isnull().sum()` → count of missing values

In [1]:
print("Shape (rows, columns):", df.shape)

Shape (rows, columns): (256, 12)

In [1]:
print("Columns:")
list(df.columns)

Columns:

In [1]:
df.dtypes

Order_ID             str
Order_Date           str
Customer_Name        str
Segment              str
City                 str
State                str
Category             str
Sub_Category         str
Product_Name         str
Price            float64
Quantity         float64
Discount         float64
dtype: object

In [1]:
df.tail()

     Order_ID  Order_Date Customer_Name      Segment      City State         Category Sub_Category      Product_Name   Price  Quantity  Discount
251  ORD-1249  2024-06-28   Customer_49     Consumer     Miami    FL        Furniture     Bookcase  Bookcase Model-8  234.19       9.0       0.1
252  ORD-1230  2024-07-01   Customer_95  Home Office   Houston    TX       Technology       Laptop    Laptop Model-2  454.43       2.0       0.0
253  ORD-1161  2024-10-08   Customer_83    Corporate  New York    NY  Office Supplies      Pen Set   Pen Set Model-3  196.77       6.0       0.0
254  ORD-1091  2024-09-10   Customer_33     Consumer   Seattle    WA  Office Supplies       Binder    Binder Model-3  301.03       7.0       0.0
255  ORD-1224  2024-05-06   Customer_45  Home Office   Seattle    WA  Office Supplies        Paper     Paper Model-9  346.67       8.0       0.0

In [1]:
# Count of missing (NaN) values in each column
df.isnull().sum()

Order_ID         0
Order_Date       0
Customer_Name    0
Segment          0
City             8
State            0
Category         0
Sub_Category     0
Product_Name     0
Price            8
Quantity         8
Discount         8
dtype: int64

## Step 3: Handle missing values

We can see that `City`, `Price`, `Quantity`, and `Discount` have some missing (NaN) values.

Strategy:
- **Price, Quantity** → numeric columns, so missing values are filled with the **median** (more robust than the mean if there are outliers)
- **Discount** → a missing value likely means "no discount was applied", so fill with `0`
- **City** → text column, so fill with `"Unknown"`

(You could also use `.dropna()` to drop entire rows with missing values, but filling is generally preferred when data is limited.)

In [1]:
# Fill Price and Quantity with median
df['Price'] = df['Price'].fillna(df['Price'].median())
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())

# Missing Discount = 0 discount
df['Discount'] = df['Discount'].fillna(0)

# Missing City = "Unknown"
df['City'] = df['City'].fillna("Unknown")

# Verify - all counts should now be 0
df.isnull().sum()

Order_ID         0
Order_Date       0
Customer_Name    0
Segment          0
City             0
State            0
Category         0
Sub_Category     0
Product_Name     0
Price            0
Quantity         0
Discount         0
dtype: int64

## Step 4: Basic operations — filtering rows, selecting columns

- **Filter rows**: keep only rows that match a condition (e.g. Category = "Technology")
- **Select columns**: view only a few specific columns

In [1]:
# Filter: only Technology category orders
tech_df = df[df['Category'] == 'Technology']
print("Technology rows:", tech_df.shape)
tech_df[['Order_ID', 'Product_Name', 'Price']].head()

Technology rows: (92, 12)
   Order_ID      Product_Name   Price
0  ORD-1000    Laptop Model-4   17.38
1  ORD-1001   Laptop Model-17  297.29
7  ORD-1007    Laptop Model-9  306.52
8  ORD-1008  Monitor Model-13  343.88
13  ORD-1013   Laptop Model-10  342.45

In [1]:
# Select: only a few columns
selected = df[['Order_ID', 'Category', 'Price', 'Quantity']]
selected.head()

   Order_ID         Category   Price  Quantity
0  ORD-1000       Technology   17.38       5.0
1  ORD-1001       Technology  297.29       1.0
2  ORD-1002        Furniture  359.43       9.0
3  ORD-1003        Furniture  173.42       3.0
4  ORD-1004  Office Supplies  303.84       1.0

## Step 5: Remove duplicate rows

`.duplicated()` flags rows that are exact repeats, and `.drop_duplicates()` removes them.

In [1]:
print("Duplicate rows before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Duplicate rows before: 6
Shape after removing duplicates: (250, 12)

## Step 6: Create a derived column — `total_amount`

A new column calculated from existing columns:

`total_amount = Price * Quantity`

In [1]:
df['total_amount'] = df['Price'] * df['Quantity']

df[['Order_ID', 'Price', 'Quantity', 'total_amount']].head()

   Order_ID   Price  Quantity  total_amount
0  ORD-1000   17.38       5.0         86.90
1  ORD-1001  297.29       1.0        297.29
2  ORD-1002  359.43       9.0       3234.87
3  ORD-1003  173.42       3.0        520.26
4  ORD-1004  303.84       1.0        303.84

## Step 7: Save the cleaned dataset to a new CSV file

In [1]:
df.to_csv("superstore_cleaned.csv", index=False)

print("Saved! Final shape:", df.shape)
df[['Price', 'Quantity', 'Discount', 'total_amount']].describe()

Saved! Final shape: (250, 13)
            Price    Quantity    Discount  total_amount
count  250.000000  250.000000  250.000000    250.000000
mean   254.798960    5.720000    0.098400   1479.963640
std    136.548012    2.849928    0.114411   1167.971672
min      6.610000    1.000000    0.000000     33.050000
25%    141.950000    3.000000    0.000000    466.787500
50%    253.130000    6.000000    0.000000   1243.875000
75%    360.655000    8.000000    0.200000   2308.590000
max    499.790000   10.000000    0.300000   4843.600000

## Summary — What we did

| Step | Task |
|---|---|
| 1 | Loaded the CSV file into a DataFrame (256 rows, 12 columns) |
| 2 | Explored the data — shape, columns, dtypes, missing values |
| 3 | Fixed missing values — Price/Quantity → median, Discount → 0, City → "Unknown" |
| 4 | Filtered rows (by Category) and selected specific columns |
| 5 | Removed 6 duplicate rows |
| 6 | Created a new column `total_amount = Price × Quantity` |
| 7 | Saved the final cleaned data to `superstore_cleaned.csv` (250 rows, 13 columns) |

**Next step:** Run this same notebook on your actual Kaggle Superstore CSV — just change the file name/path in `pd.read_csv("superstore_raw.csv")`. Column names may differ slightly (e.g. `Sales`, `Order Date` with spaces), so check `df.columns` and adjust accordingly.